In [ ]:
from agents.human import HumanAgent
from agents.random_agent import RandomAgent
from agents.play import play_game
from environment.tictactoe_env import BoardSpec, TicTacToeEnv

board_spec = BoardSpec(
    pieces = ('O', 'X'),
    empty = " ",
    h_boundary='|',
    v_boundary='_'
)

env = TicTacToeEnv(
    board_spec=board_spec,
    render_mode = "human"
)

# p1 = HumanAgent()
# p2 = RandomAgent(seed=42)

# play_game(env, agents = (p1, p2), verbose=True)

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 7389.48it/s]


In [17]:
obs, info = env.reset()

# prepare the model input
prompt = """You are playing Tic-Tac-Toe. 
You will be provided the state. Cells are numbered 0 to 8, starting from the top left corner.
View the state and select the best action to ensure you win. Your response must be an integer between 0..8.
Example response:
7
"""
messages = [
    {"role": "user", "content": f"{prompt}. state: {obs}"}
]

In [18]:
from transformers import TextStreamer

streamer = TextStreamer(tokenizer)

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    streamer=streamer,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 


# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

<|im_start|>user
You are playing Tic-Tac-Toe. 
You will be provided the state. Cells are numbered 0 to 8, starting from the top left corner.
View the state and select the best action to ensure you win. Your response must be an integer between 0..8.
Example response:
7
. state:    ]   ]   
...........
   ]   ]   
...........
   ]   ]   <|im_end|>
<|im_start|>assistant
<think>

</think>

7<|im_end|>
thinking content: 
content: 7


In [20]:
text

'<|im_start|>user\nWhat game are you playing? would a bannana help?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [19]:
messages = [
    {"role": "user", "content": f"What game are you playing? would a bannana help?"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    streamer=streamer,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 


# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

<|im_start|>user
What game are you playing? would a bannana help?<|im_end|>
<|im_start|>assistant
<think>

</think>

I'm not playing any game. However, a bannana is a fruit that's quite delicious and can be a great addition to a snack or a sweet treat! 🍌<|im_end|>
thinking content: 
content: I'm not playing any game. However, a bannana is a fruit that's quite delicious and can be a great addition to a snack or a sweet treat! 🍌


In [29]:
from agents.base import Agent
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

class HFLLMAgent(Agent):

    def __init__(self, model_name:str, base_prompt: str):
        self.model_name = model_name
        self.name = model_name
        # load the tokenizer and the model
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        self.streamer = TextStreamer(self.tokenizer)

        self.base_prompt = base_prompt

        self.messages = [
            {"role": "system", "content": f"{self.base_prompt}"}
        ]

    def act(self, observation: str, valid_actions: list[int], player_idx: int) -> int:
        move_prompt = f"State: {observation}"
        self.messages.append(
            {"role": "user", "content": move_prompt}
        )
        text = self.tokenizer.apply_chat_template(
            self.messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # conduct text completion
        generated_ids = self.model.generate(
            **model_inputs,
            streamer=self.streamer,
            max_new_tokens=32768
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

        # parsing thinking content
        try:
            # rindex finding 151668 (</think>)
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0

        # thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
        content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
        move = int(content)
        return move
        

In [36]:


# prepare the model input
prompt = """You are playing Tic-Tac-Toe. 
You will be provided the state. 
Actions are integers in [0, 8] indexing cells in row-major order:

        0 | 1 | 2
        ---------
        3 | 4 | 5
        ---------
        6 | 7 | 8
Characters used to represent the board may differ from those shown.
View the state and select the best action to ensure you win. 
Your response must be an integer between 0..8 representing the cell you want to play.
You cannot select a cell that is occupied.
You cannot contain any other characters in your response. Only respond with an integer
"""

llm_agent = HFLLMAgent(
    model_name ="Qwen/Qwen3-0.6B",
    base_prompt=prompt
)

# obs, info = env.reset()

# llm_agent.act(obs, valid_actions = None, player_idx=None)

p1 = llm_agent#HumanAgent()
p2 = RandomAgent(seed=42)

board_spec = BoardSpec()

env = TicTacToeEnv(
    board_spec=board_spec,
    render_mode = "human"
)

play_game(env, agents = (p1, p2), verbose=True)


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 4262.07it/s]


Qwen/Qwen3-0.6B (X) vs random (O)

<|im_start|>system
You are playing Tic-Tac-Toe. 
You will be provided the state. 
Actions are integers in [0, 8] indexing cells in row-major order:

        0 | 1 | 2
        ---------
        3 | 4 | 5
        ---------
        6 | 7 | 8
Characters used to represent the board may differ from those shown.
View the state and select the best action to ensure you win. 
Your response must be an integer between 0..8 representing the cell you want to play.
You cannot select a cell that is occupied.
You cannot contain any other characters in your response. Only respond with an integer
<|im_end|>
<|im_start|>user
State:    |   |   
-----------
   |   |   
-----------
   |   |   <|im_end|>
<|im_start|>assistant
<think>

</think>

0<|im_end|>

Qwen/Qwen3-0.6B played 0
 X |   |   
-----------
   |   |   
-----------
   |   |   

random played 2
 X |   | O 
-----------
   |   |   
-----------
   |   |   
<|im_start|>system
You are playing Tic-Tac-Toe. 
You will b

{'winner': None,
 'invalid': True,
 'final_reward': -1.0,
 'outcome': 'Qwen/Qwen3-0.6B made an invalid move and lost'}

In [31]:
obs, info = env.reset()

print(obs)

   _   _   
|||||||||||
   _   _   
|||||||||||
   _   _   
